# Expenses

Load Dataset

In [ ]:
import pandas as pd
import numpy as np

df5 = pd.read_excel("../Datasets/Expenses.xlsx")
print(df5.shape)
df5.head()

In [ ]:
df5.info()

Check Missing Values

In [ ]:
df5.isna().sum()

Remove Unwanted Columns

In [ ]:
df5 = df5.drop(columns=["Municipality", "Province", "Region", "ID"], errors="ignore")

Standardize Schema

In [ ]:
df5 = df5.rename(columns={
    "Expenses": "Transaction Description",
    "Category": "Category",
    "Amount": "Amount",
    "Date": "Date"
})
print("[INFO] Columns after renaming:")
print(df5.columns)

Amount column, Ensure numeric and positive:

In [ ]:
df5["Amount"] = pd.to_numeric(df5["Amount"], errors="coerce")
df5 = df5[df5["Amount"] > 0]

Category & Description

In [ ]:
df5["Transaction Description"] = df5["Transaction Description"].fillna("").astype(str)
df5["Category"] = df5["Category"].fillna("Other").astype(str)

Create TransactionID

In [ ]:
df5 = df5.reset_index(drop=True)
df5["TransactionID"] = df5.index + 1
df5["TransactionID"] = df5["TransactionID"].apply(
    lambda x: f"EXP5_{str(x).zfill(6)}"
)

Create UserID

In [ ]:
df5["UserID"] = "US005"

Create Type (Income / Expense)

In [ ]:
df5["Category"].unique()

In [ ]:
df5["Transaction Description"].unique()

In [ ]:
# Contains only expenses
df5["Type"] = "Expense"

Currency Inference

In [ ]:
# Dataset is from the Philippines:
# Region: National Capital Region
# Quezon City → Metro Manila
# Amounts like 40, 60, 199 → matches PHP
# Category naming style consistent with PH household surveys
df5["Currency"] = "PHP"

Merchant Creation

In [ ]:
df5["Merchant"] = ""

Account Name Creation

In [ ]:
df5["Account Name"] = ""

Create an LLM function

In [ ]:
# Rename existing columns
df5 = df5.rename(columns={
    "Transaction Description": "Category",   # move this into Category
    "Category": "Raw_Priority"               # store old priority column
})

# Step 2 — Ensure new columns exist properly as strings
df5["Category"] = df5["Category"].astype(str).str.strip()
df5["Raw_Priority"] = df5["Raw_Priority"].astype(str).str.strip()

In [ ]:
from groq import Groq
import pandas as pd

client = Groq(api_key="gsk_REDACTED")


In [ ]:
def generate_description(category, priority, amount):
    prompt = f"""
    You are generating a short, clear transaction description.

    Category: {category}
    Spending Priority: {priority}
    Amount: {amount}

    Generate a realistic human transaction description (max 8–12 words).
    It should sound like an actual spending note.
    Do NOT repeat the category. 
    Do NOT add quotes. 
    Keep it natural.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",    
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
        max_tokens=50
    )

    return response.choices[0].message.content.strip()


In [ ]:
df5["Transaction Description"] = df5.apply(
    lambda row: generate_description(
        row["Category"],
        row["Raw_Priority"],
        row["Amount"]
    ),
    axis=1
)

df5.head()

Final Validation

In [ ]:
print("Total Records:", df5.shape[0])
# each user transaction count
print(df5["UserID"].value_counts())

In [ ]:
df5.head()

In [ ]:
required = ["TransactionID","UserID","Date","Category","Amount","Type"]
print(df5[required].isna().sum())

Save Clean Dataset

In [ ]:
import os
os.makedirs("Tofinal", exist_ok=True)

df5.to_csv("Tofinal/Expenses.csv", index=False)
print("File saved successfully → Tofinal/Expenses.csv")